## Region Table Cleaning

In [0]:
#Display the contents of the region table in the bronze layer
#drop column _resqued_data generated during ingestion in the bronze layer
df_region = spark.table("accenture_final_project.bronze_layer.region").drop("_rescued_data")
display(df_region)


In [0]:
from pyspark.sql.types import StringType, IntegerType
from pyspark.sql import functions as F

#Schema define just to make sure the schema is correct
df_region = df_region.select(
    F.col("SalesTerritoryKey").cast(IntegerType()),
    F.col("Region").cast(StringType()),
    F.col("Country").cast(StringType()),
    F.col("Group").cast(StringType())
)

In [0]:
#Trim Spaces
df_region = df_region.select(
    *(F.trim(F.col(c)).alias(c) 
      if isinstance(c, StringType) #if column is StringType then trim spaces, else keep it unchanged
      else F.col(c) for c in df_region.columns)
)

In [0]:
#Check for duplicates
duplicates = df_region.groupBy(df_region.columns).count().filter(F.col("count") > 1)
display(duplicates)

In [0]:
#Check for missing values
missing_values = df_region.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_region.columns])
display(missing_values)


In [0]:
df_region.select("Country").distinct().display()

In [0]:
#Replace United States with USA for standardization of Country column
df_region = df_region.replace({"USA" : "United States"}, subset=["Country"])
display(df_region)

## Transform to Delta Table

In [0]:
#Save Region DataFrame as Delta Table inside the silver layer
df_region.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("accenture_final_project.silver_layer.region")
